In [7]:
!pip install ortools

# **Exercício 2 — Sudoku 3D**

Este problema deve usar a **otimização CP** (“Constraint Programming”) no **OR-Tools** e procura implementar soluções de uma generalização do problema **Sudoku**.  
Neste trabalho pretende-se generalizar o problema em várias direções:

---

### Estrutura da grelha

Em primeiro lugar, a grelha tem como parâmetro fundamental um inteiro  
\[
n \in \{3,4,...\}
\]

Fundamentalmente, a grelha passa de um quadrado com  
\[
n^2 \times n^2
\]  
células, para um **cubo tridimensional** de dimensões  
\[
n^2 \times n^2 \times n^2
\]

Cada posição na grelha é representada por um triplo de inteiros  
\[
(i, j, k) \in \{1..n^2\}^3
\]

---

### Boxes (Regiões)

Em segundo lugar, as “regiões” que a definição menciona deixam de ser linhas, colunas e “sub-grids”, para passar a ser qualquer **“box” genérica** com um número de células  
\[
\leq n^3
\]

Cada “box” é representada por um **dicionário** \( D \) que associa, no estado inicial, cada posição  
\[
(i, j, k) \in D
\]
na “box” a um valor inteiro no intervalo  
\[
\{0..n^3\}
\]

---

### Inicialização

Na inicialização da solução:
- As células associadas ao valor **0** estão livres para ser instanciadas com qualquer valor não nulo.
- Se uma célula está associada a um valor **não-nulo**, então esse valor está **fixo** e qualquer solução do problema **não o modifica**.

---

### Restrições

A solução final do problema, tal como no Sudoku original, verifica uma restrição do tipo **all-different**, que, neste caso, tem a forma:

> Dentro de uma mesma “box”, todas as células têm valores distintos no intervalo  
> \[
> \{1..n^3\}
> \]

---

### Formas de Boxes

Consideram-se neste problema duas formas básicas de “boxes”:

1. **Cubos** de \( n^3 \) células, determinados pelo seu **vértice superior, anterior e esquerdo**.  
2. **Paths**, determinados pelo seu **vértice de início**, **vértice final** e pela **ordem entre os índices dos vértices sucessivos**.

---

### Input do Problema

O *input* do problema é composto por:
- Um **conjunto de boxes**;  
- E um **conjunto de alocações de valores** a células.


## Função `solve_sudoku3d`

Esta função resolve uma generalização tridimensional do Sudoku utilizando Constraint Programming (CP) através do módulo OR-Tools.

---

### Objetivo
Encontrar uma atribuição de valores para todas as células \((i, j, k)\) de um cubo tridimensional \(n^2 \times n^2 \times n^2\), tal que:
- Cada célula tenha um valor no intervalo \(\{1..n^3\}\);
- Em cada *box* (ou “região”), todos os valores sejam distintos (`AllDifferent`);
- As células com valores fixos (`fixed_cells`) mantenham o valor inicial.

---

### Parâmetros

| Parâmetro | Tipo | Descrição |
|------------|------|------------|
| `n` | `int` | Parâmetro base da grelha (ex: `n=3` → cubo 9×9×9). |
| `boxes` | `list[list[tuple]]` | Lista de *boxes*, onde cada box é uma lista de triplos `(i, j, k)` que identificam as células que a compõem. |
| `fixed_cells` | `dict[(i,j,k) -> int]` | Dicionário de células com valores fixos (no intervalo `1..n^3`). |
| `time_limit_s` | `int` | Tempo máximo de execução (em segundos). |
| `workers` | `int` | Número de threads paralelas para o solver (default: 8). |
| `debug` | `bool` | Ativa mensagens de depuração opcionais. |

---

### Etapas principais da função

#### 1. Cálculos iniciais
- Define `n2 = n * n` → dimensão de cada eixo do cubo.
- Define `maxval = n^3` → valor máximo possível para cada célula.

---

#### 2. Verificação de entrada (`fixed_cells`)
Antes de criar o modelo, valida se os dados estão corretos:
- As posições são triplos inteiros `(i, j, k)` dentro do intervalo `[0, n²-1]`.
- Os valores estão dentro do intervalo `[1, n³]`.
- Garante que não há posições fora dos limites.

---

#### 3. Normalização e validação das `boxes`
- Cada box é verificada para garantir:
  - Que todas as posições são triplos válidos.
  - Que não há posições repetidas (usa-se um `set` para verificar duplicados).
  - Que o número de células não ultrapassa `n³`.
- Adicionalmente, verifica-se se há conflitos de valores fixos dentro da mesma box (mesmo valor em duas posições → erro).

---

#### 4. Construção do modelo CP
Cria-se uma instância `CpModel()` e adicionam-se variáveis:

\[
X_{i,j,k} \in \{1, 2, ..., n^3\}
\]

Cada posição `(i, j, k)` corresponde a uma variável inteira com domínio `1..maxval`.

---

#### 5. Inserção das restrições
- As células fixas são adicionadas com `model.Add(X[i,j,k] == valor)`.
- Para cada box, aplica-se a restrição `AddAllDifferent(vars_in_box)`, garantindo que todas as células dentro dessa box têm valores distintos.

---

#### 6. Estratégia de decisão
Define-se uma heurística de procura:
- `CHOOSE_MIN_DOMAIN_SIZE` — escolhe primeiro a variável com menos valores possíveis.
- `SELECT_MIN_VALUE` — tenta valores mais baixos primeiro.

Isto ajuda o solver a convergir mais rapidamente.

---

#### 7. Execução do solver
- Cria-se `CpSolver()`.
- Configura-se o limite de tempo (`time_limit_s`) e o número de *workers* (`num_search_workers`).
- Resolve-se o modelo com `solver.Solve(model)`.

---

#### 8. Validação da solução
Se o solver encontrar uma solução:
- Verifica novamente se todas as boxes respeitam o `AllDifferent`.
- Confirma que as células fixas não foram alteradas.
- Devolve um dicionário `{(i, j, k): valor}` com a solução final.

Se não houver solução viável:
- Retorna `None`.

---

### Valor de retorno
- **Solução encontrada:** dicionário `{(i,j,k): valor}` com a grelha resolvida.  
- **Sem solução / tempo excedido:** `None`.

---

### Resumo do fluxo

```text
Validação de input
        ↓
Verificação de boxes
        ↓
Construção do modelo CP
        ↓
Aplicação das restrições (AllDifferent + fixed)
        ↓
Definição de estratégia de decisão
        ↓
Execução do solver (ORTools)
        ↓
Validação da solução final
        ↓
Retorno da grelha resolvida (ou None)


In [ ]:
from ortools.sat.python import cp_model

def solve_sudoku3d(n, boxes, fixed_cells, debug=False):
    """
    n: inteiro (p.ex. 3)
    boxes: lista de boxes; cada box é uma lista de triplos (i,j,k) com indices 0..n2-1
    fixed_cells: dict keyed by (i,j,k) -> value in 1..n^3 (zeros shouldn't be present)
    """

    n2 = n * n
    maxval = n ** 3


    #verificação do input, das células fixas
    for pos, val in fixed_cells.items():
        if not (isinstance(pos, tuple) and len(pos) == 3):
            raise ValueError(f"fixed_cells: posição inválida: {pos}")
        if not all(isinstance(x, int) for x in pos):
            raise ValueError(f"fixed_cells: posição não inteira: {pos}")
        if not (1 <= val <= maxval):
            raise ValueError(f"fixed_cells: valor fora do intervalo 1..{maxval}: pos {pos} val {val}")
        if not all(0 <= x < n2 for x in pos):
            raise ValueError(f"fixed_cells: posição fora dos limites 0..{n2-1}: {pos}")

    # verificação das boxes e remove repetidos
    norm_boxes = []
    for b_idx, box in enumerate(boxes):
        if not isinstance(box, (list, tuple)):
            raise ValueError(f"box {b_idx} não é lista/tupla")
        # garantir triplos e dentro dos limites
        seen = set() #usa-se um set para ser mais eficiente para verificar se já foi observado
        norm_box = []
        for pos in box:
            if not (isinstance(pos, tuple) and len(pos) == 3 and all(isinstance(x, int) for x in pos)):
                raise ValueError(f"box {b_idx} posição inválida: {pos}")
            if not all(0 <= x < n2 for x in pos):
                raise ValueError(f"box {b_idx} posição fora do limite: {pos}")
            if pos in seen:
                # ignora duplicado (ou poderias lançar erro)
                continue
            seen.add(pos)
            norm_box.append(pos)
        if len(norm_box) > maxval:
            raise ValueError(f"box {b_idx} tem {len(norm_box)} células > max possível {maxval}")
        norm_boxes.append(norm_box)

    # detectar conflitos imediatos em fixed_cells por box, ou seja, se há valores
    for b_idx, box in enumerate(norm_boxes):
        value_positions = {}
        for pos in box:
            if pos in fixed_cells:
                val = fixed_cells[pos]
                if val in value_positions:
                    # conflito: mesmo valor fixo repetido na mesma box
                    raise ValueError(f"Conflito: valor fixo {val} aparece em {value_positions[val]} e em {pos} dentro da box {b_idx}")
                value_positions[val] = pos


    model = cp_model.CpModel()

    X = {}
    for i in range(n2):
        for j in range(n2):
            for k in range(n2):
                X[(i,j,k)] = model.NewIntVar(1, maxval, f'X_{i}_{j}_{k}')

    # colocar as fixed_cells
    for (i,j,k), val in fixed_cells.items():
        model.Add(X[(i,j,k)] == int(val))

    # AllDifferent por box (só se a box tiver >1 variável)
    for b_idx, box in enumerate(norm_boxes):
        if len(box) <= 1:
            continue
        vars_in_box = [X[pos] for pos in box]
        model.AddAllDifferent(vars_in_box)

    all_vars = list(X.values())
    model.AddDecisionStrategy(
        all_vars,
        cp_model.CHOOSE_MIN_DOMAIN_SIZE,
        cp_model.SELECT_MIN_VALUE
    )

    solver = cp_model.CpSolver()

    res = solver.Solve(model)

    if res in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        sol = {pos: solver.Value(var) for pos, var in X.items()}
        for b_idx, box in enumerate(norm_boxes):
            vals = [sol[pos] for pos in box]
            if len(vals) != len(set(vals)):
                raise RuntimeError(f"Solução inválida: duplicado na box {b_idx}")
        for pos, val in fixed_cells.items():
            if sol[pos] != val:
                raise RuntimeError(f"Solução alterou um fixed cell: {pos} tinha {val} e sol deu {sol[pos]}")

        return sol
    else:
        if debug:
            print("Solver status:", res)
        return None

## Funções para mostrar o cubo

In [ ]:
import pandas as pd
from itertools import cycle

BOX_COLORS = [
    "\033[38;5;208m",  # laranja
    "\033[38;5;45m",   # azul claro
    "\033[38;5;118m",  # verde
    "\033[38;5;207m",  # rosa
    "\033[38;5;226m",  # amarelo
    "\033[38;5;33m",   # azul escuro
    "\033[38;5;99m",   # lilás
    "\033[38;5;160m",  # vermelho
    "\033[38;5;244m",  # cinza
    "\033[38;5;82m",   # verde forte
    "\033[38;5;196m",  # vermelho vivo
    "\033[38;5;202m",  # laranja escuro
    "\033[38;5;40m",   # verde mar
    "\033[38;5;21m",   # azul aço
    "\033[38;5;128m",  # roxo
    "\033[38;5;165m",  # magenta
    "\033[38;5;220m",  # dourado
    "\033[38;5;15m",   # branco
    "\033[38;5;237m",  # cinza escuro
    "\033[38;5;51m",   # azul turquesa
    "\033[38;5;141m",  # violeta
    "\033[38;5;192m",  # rosa claro
    "\033[38;5;1m",    # castanho
    "\033[38;5;10m",   # verde claro
    "\033[38;5;200m",  # cor de rosa choque
    "\033[38;5;135m",  # azul violeta
    "\033[38;5;214m",  # cor de pêssego
]
RESET = "\033[0m"

def color_boxes(sol, boxes, n2):
    """
    Mostra o cubo fatiado (por k) com cores diferentes por box.
    """
    color_cycle = cycle(BOX_COLORS)
    box_colors = {}

    # atribui uma cor a cada box (ordem dos boxes)
    for idx, box in enumerate(boxes):
        box_colors[idx] = next(color_cycle)

    # cria um mapa (i,j,k) -> cor
    cell_color = {}
    for b_idx, box in enumerate(boxes):
        color = box_colors[b_idx]
        for (i,j,k) in box:
            cell_color[(i,j,k)] = color

    # DataFrame da solução
    df = pd.DataFrame([
        {"i":i, "j":j, "k":k, "val":v}
        for (i,j,k), v in sol.items()
    ])

    # imprimir cada fatia (k fixo)
    for k_view in range(n2):
        print(f"\n\033[1mFatia k={k_view} (i x j):\033[0m")
        df_slice = df[df["k"] == k_view]
        pivot_slice = df_slice.pivot(index='i', columns='j', values='val')

        # imprimir com cores
        for i in range(n2):
            row_str = ""
            for j in range(n2):
                val = pivot_slice.loc[i, j]
                color = cell_color.get((i,j,k_view), "")
                row_str += f"{color}{val:3d}{RESET} "
            print(row_str)

def make_subcube(x,y,z,n):
  return [(i,j,k) for i in range(x,x+n)
                  for j in range(y,y+n)
                  for k in range(z,z+n)]

## Exemplo Simples (Cubo 9x9x9)

In [ ]:
n = 3                # dimensão de bloco = n (3); n^2 = 9; n^3 = 27
n2 = n * n           # 9
maxval = n ** 3      # 27

boxes = []
for z in [0,3,6]:
  for y in [0,3,6]:
    for x in [0,3,6]:
      boxes.append(make_subcube(x, y, z, n))

# um path diagonal (9 células) do (0,0,0) até (8,8,8)
diag_path = [(i, i, i) for i in range(n2)]
boxes.append(diag_path)

# fixed_cells: escolhas simples, garantidas sem conflitos dentro das boxes
fixed = {}
# no cubo 0 (valores distintos, em 1..27)
fixed[(0,0,0)] = 1
fixed[(0,1,0)] = 2
fixed[(1,0,0)] = 3

# no cubo 1 (centro)
fixed[(3,3,3)] = 4
fixed[(4,3,3)] = 5

# no cubo 2 (canto oposto)
fixed[(6,6,6)] = 6

# no final da diagonal path (não conflita com números anteriores)
fixed[(8,8,8)] = 7

print(f"Testando n={n} (grid {n2}x{n2}x{n2}), maxval={maxval}")
print(f"Boxes: {len(boxes)} (3 cubos + 1 path diagonal)")
print("Fixed cells (exemplo):", fixed)

sol = solve_sudoku3d(n, boxes, fixed)

if sol is None:
    print("Nenhuma solução encontrada dentro do limite / problema possivelmente demasiado difícil.")
else:
  color_boxes(sol,boxes,n2)

Testando n=3 (grid 9x9x9), maxval=27
Boxes: 28 (3 cubos + 1 path diagonal)
Fixed cells (exemplo): {(0, 0, 0): 1, (0, 1, 0): 2, (1, 0, 0): 3, (3, 3, 3): 4, (4, 3, 3): 5, (6, 6, 6): 6, (8, 8, 8): 7}

Fatia k=0 (i x j):
  1   2  12  13   1  17  19  25   3 
  3  21   5  19  14   7  16   9   4 
 10   8  16  12  24  16  24  15   7 
 24  22  16  17  10  26  13  25   3 
  8  14  12  14  25  15  24   5  27 
 15   9  23  23   2  24  19   8  21 
 26  24  25  10  27  16  23   2  22 
 15  11  19   5  24  12  25  27   9 
 27   6  20  22   6   2  13  20  16 

Fatia k=1 (i x j):
  7  20  24  10   2  15   1  22   6 
 25  15  13   3   4   6  20  18  11 
 27   4  23  23  18  11   5  17  12 
  1   3  11  21  13   9   2   6  23 
 13  25  18  16   4  22  14  26  12 
 26  27   4   5  12  18  20   9  18 
 21   8   1   3  25  11   8  15   3 
  9  10  18  18  17  19  18  10   4 
  2  17   4  13  14  26  14   5  19 

Fatia k=2 (i x j):
 26  17   9   9  25   5  27  13   2 
 22  18   6  26  20  21  26   8  23 
 11

In [ ]:
n = 3                 # dimensão de bloco = n (3)
n2 = n * n            # 9
maxval = n ** 3       # 27

# -------------------------------------------
# Criação das boxes
# -------------------------------------------

boxes = []
# Cria os 27 subcubos 3x3x3 dentro do cubo 9x9x9
for z in [0, 3, 6]:
    for y in [0, 3, 6]:
        for x in [0, 3, 6]:
            boxes.append(make_subcube(x, y, z, n))

# Box extra: caminho diagonal principal (0,0,0) → (8,8,8)
diag_main = [(i, i, i) for i in range(n2)]
boxes.append(diag_main)

linha_sup_esq_cima = [(0,0,k) for k in range(n2)]
boxes.append(linha_sup_esq_cima)

linha_inf_dir_baixo = [(8,8,k) for k in range(n2)]
boxes.append(linha_inf_dir_baixo)

# Box extra: diagonal inversa (0,8,0) → (8,0,8)
diag_inv = [(i, 8 - i, i) for i in range(n2)]
boxes.append(diag_inv)

# -------------------------------------------
# Células fixas (sem conflitos)
# -------------------------------------------
fixed = {
    # canto frontal superior
    (0, 0, 0): 1,
    (1, 0, 0): 2,
    (2, 0, 0): 3,
    (0, 1, 0): 4,
    (0, 2, 0): 5,

    # centro aproximado
    (4, 4, 4): 6,
    (4, 5, 4): 7,
    (5, 4, 4): 8,

    # plano z = 3
    (0, 0, 3): 9,
    (3, 3, 3): 10,
    (6, 6, 3): 11,

    # plano z = 6
    (1, 1, 6): 12,
    (2, 2, 6): 13,
    (3, 3, 6): 14,
    (4, 4, 6): 15,

    # canto oposto (final da diagonal principal)
    (8, 8, 8): 16,

    # diagonal inversa
    (0, 8, 0): 17,
    (8, 0, 8): 18,

    # posições aleatórias extra
    (5, 1, 2): 22,
    (1, 5, 7): 23,
    (7, 3, 1): 24,
    (8, 4, 5): 25,
    (3, 8, 2): 26,
    (2, 6, 8): 27
}

sol = solve_sudoku3d(n, boxes, fixed)

if sol is None:
    print("Nenhuma solução encontrada dentro do limite ou problema demasiado complexo.")
else:
    print("Solução encontrada! Validando e colorindo boxes...\n")
    color_boxes(sol, boxes, n2)

Solução encontrada! Validando e colorindo boxes...


Fatia k=0 (i x j):
  1   4   5   2   8   4  23  12  17 
  2  21  18  22  27  26  10  13  18 
  3  25   9   3  15   6  15  16  11 
 23  12   5   4  19  12  11   2   6 
 27  25  13   7  13   1  25  21  18 
 20   3  10   2   9  15  27  19  24 
 22   8  17  18  15  19  13  12   8 
 15  23   1   2   1   9   3   4  25 
 24   3   5   4  10  23  27  19  17 

Fatia k=1 (i x j):
 17  26  11  18  23  24   5  14   3 
 27  22  13   9  11  16   2  26  20 
 24  16  10  13  25  21   4   1  19 
 14   2  24  20  17   5  20  14   9 
 15  17  21  21  22  14  15   8  23 
  1   9   4  10  11   6  16  13  22 
  4  21  20  20  11  22  14  15   5 
 16   6  26  24  13  17   7   6  23 
 12  18   7  21   6  26  21  18  20 

Fatia k=2 (i x j):
 23  20   6   5  19  17   8  27  22 
 14  12   7  12   1  20   9  24   7 
 15   8  19  14   7  10  25   6  21 
 26   6  19  26  16  18   3  17  26 
 18   8  11   3  23  24  12  10   7 
 16  22   7   8  27  25   1   5   4 


## Exemplo Médio (Cubo 16x16x16)

In [ ]:
# Exemplo para n = 4
n = 4
n2 = n * n
maxval = n ** 3

print(f"Criando exemplo para n={n} (grid {n2}x{n2}x{n2}), maxval={maxval}")

# Definir algumas boxes (cubos) para n=4
boxes_n4 = []

for z in [0,4,8,12]:
  for y in [0,4,8,12]:
    for x in [0,4,8,12]:
      boxes_n4.append(make_subcube(x, y, z, n))

# Adicionar um path diagonal para n=4
diag_path_n4 = [(i, i, i) for i in range(n2)]
boxes_n4.append(diag_path_n4)

# Definir algumas fixed_cells para n=4
fixed_n4 = {}
fixed_n4[(0,0,0)] = 1 # No primeiro cubo
fixed_n4[(0,1,0)] = 2
fixed_n4[(1,0,0)] = 3
fixed_n4[(n,n,n)] = 4 # No cubo do meio
fixed_n4[(n+1,n,n)] = 5
fixed_n4[(n+1, n+1, n+1)] = 6 # Na diagonal e no cubo do meio
fixed_n4[(n2-1, n2-1, n2-1)] = 7 # No final da diagonal
fixed_n4[(0, 0, 4)] = 8
fixed_n4[(4, 0, 0)] = 9
fixed_n4[(0, 4, 0)] = 10
fixed_n4[(8, 8, 8)] = 11
fixed_n4[(12, 12, 12)] = 12


print(f"Número de boxes criadas: {len(boxes_n4)}")
print("Fixed cells (exemplo):", fixed_n4)

# Resolver para n=4
# Pode ser necessário aumentar o time_limit_s para problemas maiores
sol_n4 = solve_sudoku3d(n, boxes_n4, fixed_n4)

if sol_n4 is None:
    print("Nenhuma solução encontrada dentro do limite / problema possivelmente demasiado difícil para n=4.")
else:
    print("\nSolução encontrada para n=4! Algumas células (pos -> val):")
    color_boxes(sol_n4,boxes_n4,n2)

Criando exemplo para n=4 (grid 16x16x16), maxval=64
Número de boxes criadas: 65
Fixed cells (exemplo): {(0, 0, 0): 1, (0, 1, 0): 2, (1, 0, 0): 3, (4, 4, 4): 4, (5, 4, 4): 5, (5, 5, 5): 6, (15, 15, 15): 7, (0, 0, 4): 8, (4, 0, 0): 9, (0, 4, 0): 10, (8, 8, 8): 11, (12, 12, 12): 12}

Solução encontrada para n=4! Algumas células (pos -> val):

Fatia k=0 (i x j):
  1   2  15  53  10  45  25  43  40  27  35  29  40  59  23  49 
  3   8  16  12  13  41  31  55  61  58  62  11   2  58   7   8 
 43  38  21  52  62   8  32  15  21   5  36   6  60  26   5  17 
 63  42  62  48   1  52  14   6  55   9  18  59   9  20  27  12 
  9  44  19  28  31  10  30   8   9  60  54  41   3  16   1  12 
 62  64  41   7  16  32  57  24   6   7  52  42  61  20  17  35 
  5  48  24  50  36   1  41  45  15  33  64  11  15  53  31  19 
 18   3  33  58  12  40   6  50  34  18  23  53   9  21  26  36 
 62  20  45   6  64  38  19  47  22  31   1  27  42  34  20  19 
 42  47  17  64  16   5  34  50  59  53  13  19  59  2

## Exemplo Grande (Cubo 25x25x25)

In [ ]:
n = 5
n2 = n * n            # 25
maxval = n ** 3       # 125

print(f"Criando exemplo para n={n} (grid {n2}x{n2}x{n2}), maxval={maxval}")

# -------------------------------------------
# Criação das boxes
# -------------------------------------------
boxes_n5 = []

# cria os 125 subcubos (5x5x5) dentro do cubo 25x25x25
for z in range(0, n2, n):
    for y in range(0, n2, n):
        for x in range(0, n2, n):
            boxes_n5.append(make_subcube(x, y, z, n))

# adicionar uma diagonal principal (0,0,0) → (24,24,24)
diag_path_n5 = [(i, i, i) for i in range(n2)]
boxes_n5.append(diag_path_n5)

# adicionar uma diagonal inversa (0,24,0) → (24,0,24)
diag_inv_n5 = [(i, n2 - 1 - i, i) for i in range(n2)]
boxes_n5.append(diag_inv_n5)
# -------------------------------------------
# Definição de algumas células fixas
# -------------------------------------------
fixed_n5 = {}

# canto frontal superior
fixed_n5[(0,0,0)] = 1
fixed_n5[(1,0,0)] = 2
fixed_n5[(2,0,0)] = 3
fixed_n5[(3,0,0)] = 4
fixed_n5[(4,0,0)] = 5

# linha principal do meio (centro aproximado)
fixed_n5[(12,12,12)] = 6
fixed_n5[(13,12,12)] = 7
fixed_n5[(12,13,12)] = 8
fixed_n5[(12,12,13)] = 9
fixed_n5[(11,11,11)] = 10

# plano z = 5
fixed_n5[(0,0,5)] = 11
fixed_n5[(5,5,5)] = 12
fixed_n5[(10,10,5)] = 13
fixed_n5[(15,15,5)] = 14
fixed_n5[(20,20,5)] = 15

# plano z = 20
fixed_n5[(1,1,20)] = 16
fixed_n5[(5,10,20)] = 17
fixed_n5[(10,5,20)] = 18
fixed_n5[(15,10,20)] = 19
fixed_n5[(20,15,20)] = 20

# alguns pontos aleatórios na diagonal inversa
fixed_n5[(0,24,0)] = 21
fixed_n5[(5,19,5)] = 22
fixed_n5[(10,14,10)] = 23
fixed_n5[(15,9,15)] = 24
fixed_n5[(24,0,24)] = 25

print(f"Número de boxes criadas: {len(boxes_n5)} (125 cubos + 3 extras)")
print(f"Número de células fixas: {len(fixed_n5)}\n")


sol_n5 = solve_sudoku3d(n, boxes_n5, fixed_n5)

if sol_n5 is None:
    print("Nenhuma solução encontrada dentro do limite / problema possivelmente demasiado difícil para n=5.")
else:
    print("\nSolução encontrada para n=5! Algumas células (pos -> val):")
    color_boxes(sol_n5, boxes_n5, n2)


Criando exemplo para n=5 (grid 25x25x25), maxval=125
Número de boxes criadas: 127 (125 cubos + 3 extras)
Número de células fixas: 25


Solução encontrada para n=5! Algumas células (pos -> val):

Fatia k=0 (i x j):
  1 121 116 111 106 125 120 115 110 105 125 120 115 110 105 125 120 115 110 105 125 120 115 110  21 
  2  97  92  87  82 100  95  90  85  80 100  95  90  85  80 100  95  90  85  80 101  96  91  86  81 
  3  73  68  63  58  75  70  65  60  55  75  70  65  60  55  75  70  65  60  55  76  71  66  62  56 
  4  49  44  39  34  50  45  40  35  30  50  45  40  35  30  50  45  40  35  30  51  46  42  36  31 
  5  25  20  15  10  25  20  15  10   5  25  20  15  10   5  25  20  15  10   5  26  22  15  10   5 
125 120 115 110 105 125 120 115 110 105 125 120 115 110 105 125 120 115 110 105 125 120 115 110 105 
100  95  90  85  80 100  95  90  85  80 100  95  90  85  80 100  95  90  85  80 100  95  90  85  80 
 75  70  65  60  55  75  70  65  60  55  75  70  65  60  55  75  70  65  60  55

## Exemplo Gigante (Cubo 36x36x36)

In [ ]:
n = 6
n2 = n * n            # 36
maxval = n ** 3       # 216

print(f"Criando exemplo para n={n} (grid {n2}x{n2}x{n2}), maxval={maxval}")

# -------------------------------------------
# Criação das boxes
# -------------------------------------------
boxes_n6 = []

# cria os subcubos 6x6x6 ao longo do cubo 36x36x36
for z in range(0, n2, n):
    for y in range(0, n2, n):
        for x in range(0, n2, n):
            boxes_n6.append(make_subcube(x, y, z, n))

# adicionar paths e camadas extras
diag_path_n6 = [(i, i, i) for i in range(n2)]                     # diagonal principal
boxes_n6.append(diag_path_n6)

diag_inv_n6 = [(i, n2 - 1 - i, i) for i in range(n2)]             # diagonal inversa
boxes_n6.append(diag_inv_n6)

# -------------------------------------------
# Células fixas (valores de exemplo)
# -------------------------------------------
fixed_n6 = {}

# canto frontal superior
fixed_n6[(0,0,0)] = 1
fixed_n6[(1,0,0)] = 2
fixed_n6[(2,0,0)] = 3
fixed_n6[(3,0,0)] = 4
fixed_n6[(4,0,0)] = 5
fixed_n6[(5,0,0)] = 6

# zona central aproximada
fixed_n6[(18,18,18)] = 7
fixed_n6[(19,18,18)] = 8
fixed_n6[(18,19,18)] = 9
fixed_n6[(18,18,19)] = 10
fixed_n6[(17,17,17)] = 11

# plano z = 6
fixed_n6[(0,0,6)] = 12
fixed_n6[(6,6,6)] = 13
fixed_n6[(12,12,6)] = 14
fixed_n6[(18,18,6)] = 15
fixed_n6[(24,24,6)] = 16
fixed_n6[(30,30,6)] = 17

# plano z = 30
fixed_n6[(0,0,30)] = 18
fixed_n6[(6,6,30)] = 19
fixed_n6[(12,12,30)] = 20
fixed_n6[(18,18,30)] = 21
fixed_n6[(24,24,30)] = 22
fixed_n6[(30,30,30)] = 23

# alguns pontos na diagonal inversa
fixed_n6[(0,35,0)] = 24
fixed_n6[(6,29,6)] = 25
fixed_n6[(12,23,12)] = 26
fixed_n6[(18,17,18)] = 27
fixed_n6[(24,11,24)] = 28
fixed_n6[(30,5,30)] = 29
fixed_n6[(35,0,35)] = 30

print(f"Número de boxes criadas: {len(boxes_n6)} (216 cubos + 3 extras)")
print(f"Número de células fixas: {len(fixed_n6)}\n")

sol_n6 = solve_sudoku3d(n, boxes_n6, fixed_n6)

if sol_n6 is None:
    print("Nenhuma solução encontrada dentro do limite (esperado, pois n=6 é enorme).")
else:
    print("\nSolução encontrada para n=6! Algumas células (pos -> val):")
    color_boxes(sol_n6, boxes_n6, n2)

Criando exemplo para n=6 (grid 36x36x36), maxval=216
Número de boxes criadas: 218 (216 cubos + 3 extras)
Número de células fixas: 30

